In [ ]:
# ============================================================
# UNIVERSAL INFERENCE SCRIPT
# - Descomprime el ZIP del modelo
# - Muestra hallazgos/resúmenes del entrenamiento (si existen en el ZIP)
# - Predice sobre T_new_final.csv
# - Guarda: T_new_predicciones.csv + applied_inference_policy.json
# ============================================================

import os, json, zipfile, glob, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

# -------- Config mínima (puedes dejar MODEL_ZIP=None para autodetección) ----------
NEW_CSV   = "T_new_final.csv"
MODEL_ZIP = None  # p.ej.: "svm_multiclase_artifacts_bundle.zip" ; None => autodetecta *.zip
OUTDIR    = Path("inference_outputs")

# -------- Utilidades -------------------------------------------------------------
def _find_zip(path_pattern="*.zip"):
    zips = sorted([p for p in Path(".").glob(path_pattern) if p.is_file()])
    if not zips:
        raise FileNotFoundError("No se encontró ningún ZIP de modelo en el directorio actual.")
    if len(zips) > 1:
        print("[Aviso] Se encontraron varios ZIP. Usando el más reciente por mtime.")
        zips.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return zips[0]

def _unpack_zip(zip_path: Path, unpack_dir: Path):
    unpack_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(unpack_dir)
    return unpack_dir

def _find_model_file(unpack_dir: Path):
    cand = []
    for ext in ("*.joblib", "*.pkl"): cand += list(unpack_dir.glob(ext))
    if not cand:
        for ext in ("**/*.joblib", "**/*.pkl"): cand += list(unpack_dir.glob(ext))
    if not cand:
        raise FileNotFoundError("No se encontró archivo de modelo (.joblib/.pkl) dentro del ZIP.")
    preferred = [
        "modelo_svm.pkl", "modelo_arbol.pkl", "modelo_knn.pkl", "modelo_reg_logistica.pkl",
        "adaboost_best_model.joblib", "bagging_knn_best_model.joblib"
    ]
    for name in preferred:
        hit = [p for p in cand if p.name == name]
        if hit: return hit[0]
    cand.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cand[0]

def _load_model(model_path: Path):
    return joblib.load(model_path)

def _get_final_estimator(model):
    if hasattr(model, "named_steps") and isinstance(model.named_steps, dict):
        last_key = list(model.named_steps.keys())[-1]
        return model.named_steps[last_key]
    return model

def _get_classes(model):
    for obj in (model, _get_final_estimator(model)):
        if hasattr(obj, "classes_"):
            return np.array(getattr(obj, "classes_"))
    return None

def _predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    raise AttributeError("El modelo no expone predict_proba(). Revisa cómo fue entrenado.")

def _read_inference_policy(unpack_dir: Path):
    cand = list(unpack_dir.glob("inference_policy.json")) or list(unpack_dir.glob("**/inference_policy.json"))
    if cand:
        try:
            with open(cand[0], "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return None

def _choose_pos_label(classes):
    if classes is None or len(classes) != 2: return None
    for c in classes:
        if (isinstance(c, (int, np.integer, float, np.floating)) and float(c) == 1.0) or \
           (isinstance(c, str) and str(c).strip() == "1"):
            return c
    return sorted(list(classes))[-1]

# -------- Carga de archivos ------------------------------------------------------
if MODEL_ZIP is None:
    zip_path = _find_zip("*.zip")
else:
    zip_path = Path(MODEL_ZIP)
if not zip_path.exists():
    raise FileNotFoundError(f"No existe el ZIP indicado: {zip_path}")

if not Path(NEW_CSV).exists():
    raise FileNotFoundError(f"No existe el CSV de nuevos datos: {NEW_CSV}")

print(f"Usando ZIP de modelo: {zip_path}")
print(f"Usando datos nuevos:  {NEW_CSV}")

UNPACK_DIR = Path("inference_unpack")
if UNPACK_DIR.exists():
    for p in UNPACK_DIR.rglob("*"):
        try: p.unlink()
        except Exception: pass
    try: UNPACK_DIR.rmdir()
    except Exception: pass
UNPACK_DIR.mkdir(exist_ok=True)

_unpack_zip(zip_path, UNPACK_DIR)
model_path = _find_model_file(UNPACK_DIR)
print(f"Modelo encontrado: {model_path}")

# -------- Mostrar hallazgos del entrenamiento -----------------------------------
def _read_json_if_exists(p: Path):
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def _compact_dict(d, max_items=15):
    if not isinstance(d, dict): return d
    out, items = {}, list(d.items())
    for i, (k, v) in enumerate(items):
        if i >= max_items:
            out["…"] = f"(+{len(items)-max_items} claves)"
            break
        out[k] = v
    return out

candidate_jsons = [
    "svm_resumen.json", "resumen_metricas.json", "reg_resumen.json",
    "adaboost_best_params.json", "bagging_knn_best_params.json",
    "expected_columns.json"
]
found_info = {}
for name in candidate_jsons:
    p = next(UNPACK_DIR.rglob(name), None)
    if p and p.exists():
        found_info[name] = _read_json_if_exists(p)

report_txt = None
rep_path = next(UNPACK_DIR.rglob("classification_report.txt"), None)
if rep_path and rep_path.exists():
    try:
        with open(rep_path, "r", encoding="utf-8") as f:
            report_txt = f.read()
    except Exception:
        pass

print("\n🧾 Hallazgos del entrenamiento (detectados en el ZIP):")
if not found_info and not report_txt:
    print("  (No se encontraron resúmenes/parametrías estándar en el bundle.)")
else:
    for k, v in found_info.items():
        if v is None: continue
        print(f"\n— {k}:")
        print(json.dumps(_compact_dict(v), ensure_ascii=False, indent=2))
    if report_txt:
        print("\n— classification_report.txt (primeras líneas):")
        print("\n".join(report_txt.strip().splitlines()[:20]))

# -------- Cargar modelo / policy y nuevos datos ---------------------------------
model = _load_model(model_path)
classes_ = _get_classes(model)
policy = _read_inference_policy(UNPACK_DIR)
X_new = pd.read_csv(NEW_CSV)

# -------- Predicción de probabilidades ------------------------------------------
probs = _predict_proba(model, X_new)
if classes_ is None:
    classes_ = np.arange(probs.shape[1])

K = len(classes_)
print(f"\nClases detectadas: {list(classes_)}")

# -------- Decisión de inferencia (policy o fallback) ----------------------------
applied_policy = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "source_zip": str(zip_path),
    "decision": None,
    "classes": [str(c) for c in classes_]
}

if K == 2:
    if policy and policy.get("task") == "binary" and policy.get("decision", {}).get("type") == "threshold":
        pos_label = policy["decision"].get("pos_label") or _choose_pos_label(classes_)
        try:
            idx_pos = list(classes_).index(pos_label)
        except ValueError:
            idx_pos = [str(c).strip() for c in classes_].index(str(pos_label).strip())
            pos_label = classes_[idx_pos]
        alpha = float(policy["decision"].get("alpha", 0.5))
        criterio = policy["decision"].get("criterion", "f1")
        applied_policy["decision"] = {"type": "threshold", "alpha": alpha, "pos_label": str(pos_label),
                                      "criterion": criterio, "origin": "inference_policy.json"}
    else:
        pos_label = _choose_pos_label(classes_)
        idx_pos = list(classes_).index(pos_label)
        alpha = 0.5
        applied_policy["decision"] = {"type": "threshold", "alpha": alpha, "pos_label": str(pos_label),
                                      "criterion": "default_0.5", "origin": "fallback"}

    p_pos = probs[:, idx_pos]
    y_pred01 = (p_pos >= alpha).astype(int)
    neg_label = [c for c in classes_ if c != pos_label][0]
    mapped = np.where(y_pred01 == 1, pos_label, neg_label)

    out = X_new.copy()
    out["score"] = p_pos
    out["y_pred"] = mapped

else:
    if policy and policy.get("task") == "multiclass" and policy.get("decision", {}).get("type") == "argmax_proba":
        applied_policy["decision"] = {"type": "argmax_proba", "origin": "inference_policy.json"}
    else:
        applied_policy["decision"] = {"type": "argmax_proba", "origin": "fallback"}

    out = X_new.copy()
    for i, c in enumerate(classes_):
        out[f"score_{c}"] = probs[:, i]
    y_idx = np.argmax(probs, axis=1)
    out["y_pred"] = np.array(classes_)[y_idx]

# -------- Guardar resultados ----------------------------------------------------
OUTDIR.mkdir(parents=True, exist_ok=True)
pred_csv = OUTDIR / "T_new_predicciones.csv"
out.to_csv(pred_csv, index=False)

with open(OUTDIR / "applied_inference_policy.json", "w", encoding="utf-8") as f:
    json.dump(applied_policy, f, ensure_ascii=False, indent=2)

print("\n✅ Listo.")
print(f"- Predicciones: {pred_csv.resolve()}")
print(f"- Política aplicada: {(OUTDIR / 'applied_inference_policy.json').resolve()}")


In [ ]:
import json, pathlib

p = pathlib.Path("inference_outputs/applied_inference_policy.json")
with open(p, "r", encoding="utf-8") as f:
    data = json.load(f)
print(json.dumps(data, indent=2, ensure_ascii=False))


In [ ]:
# === Limpiar artefactos de inferencia ===
from pathlib import Path
import shutil

def safe_rmtree(p: Path):
    if not p.exists():
        print(f"⏭️  {p}/ no existe, nada que borrar.")
        return
    if not p.is_dir():
        print(f"⚠️  {p} existe pero no es carpeta. No se borra.")
        return
    if p.resolve().name not in {"inference_outputs", "inference_unpack"}:
        print(f"🛑 Salvaguarda: {p} no es una carpeta esperada. Aborto.")
        return
    shutil.rmtree(p)
    print(f"🧹 Borrado completo: {p}/")

for folder in ["inference_outputs", "inference_unpack"]:
    safe_rmtree(Path(folder))

# (Opcional) recrear vacías
for folder in ["inference_outputs", "inference_unpack"]:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print(f"📁 Recreada vacía: {folder}/")
